In [14]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def analyze_manager_report(file_path):
    # 1. 加载生成的 Markdown 报告
    loader = UnstructuredMarkdownLoader(file_path)
    docs = loader.load()
    report_content = docs[0].page_content

    # 2.  Prompt 模板
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是一位资深的基金评价专家。请根据提供的基金经理报告，生成一份精炼的分析报告。
        要求：
        1. 分析部分的字数严格控制在 250 字以内。
        2. 结构化涵盖：个人特征总结（学历、专业、从业年限、证书、履历等），关注是否名校毕业、是否有海外工作经验或海归、是否是经济金融相关专业毕业，若存在以上情况则应在报告中指出，不存在或无相关数据则不用指明。这部分若有多个基金经理则每人约25字，若只有一个基金经理则约50字。
        3. 重点结构化涵盖：基金经理管理的其他基金近一年周度的多维度的业绩排名表现。至少应占据报告的 50% 字数。
        4. 评价语言需专业、客观，业绩表现方面的观点需有数据作为支持。"""),
        ("user", "以下是基金经理详细报告内容：\n\n{content}")
    ])

    # 3. 初始化模型
    llm = ChatOpenAI(
            model="deepseek-reasoner",
            openai_api_key="sk-84ec55da76944a109524df873d20d975",
            openai_api_base="https://api.deepseek.com",
        )

    # 4. 构建分析链
    chain = prompt | llm | StrOutputParser()

    # 5. 执行分析
    analysis_result = chain.invoke({"content": report_content})
    return analysis_result

In [15]:
# 使用示例
result = analyze_manager_report("000005_Manager_Report.md")
print(result)

### 基金经理分析报告

**基金经理个人特征**
轩璇女士，工学硕士，13年从业经验，曾任易方达基金研究员。吴翠女士，硕士学历，14年从业经验，曾任信用评级机构副总经理，具备扎实的信用研究背景。

**所管理基金业绩多维排名分析**
近一年数据显示，两位基金经理管理的基金业绩分化显著。轩璇管理的产品中，纯债基金表现突出，例如**嘉实致信**（009643）在平均收益、夏普比率、选股能力上持续排名行业前20%以内（如平均收益分位多低于20%），体现了优秀的绝对收益与风险调整后收益能力。然而，其管理的**嘉实同舟**等产品（018562）同类排名则多处于后50%，平均收益分位约40%，Alpha与选股能力分位普遍在80%左右，超额收益能力较弱。

吴翠单独管理的**嘉实新添丰**（004916）与**嘉实新起点**（001688）各维度排名整体靠后。**嘉实新添丰**的平均收益率分位长期在70%左右，夏普比率分位多在40%-60%区间，风险收益性价比一般；**嘉实新起点**的波动率分位持续高于98%，显示组合稳定性在同业中处于末端。两人共同管理的**嘉实致信**业绩表现与轩璇单独管理时一致，排名领先。

**总结**
综合来看，轩璇在纯债投资领域展现出较强且稳定的主动管理能力，但管理的部分偏债混合产品表现欠佳。吴翠在管的两只灵活配置型基金，在收益获取和风险控制方面的同业排名均不具优势。


In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def analyze_manager_report(file_path):
    # 1. 加载生成的 Markdown 报告
    loader = UnstructuredMarkdownLoader(file_path)
    docs = loader.load()
    report_content = docs[0].page_content

    # 2. 针对新需求优化的 Prompt 模板
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是一位资深的基金评价专家。请根据提供的报告，完成以下维度的穿透分析，字数严格控制在 250 字以内：

1. 经理画像 (占比20%)：
   请精炼总结学历、专业、履历及证书等个人信息。必须明确指出其身份标签：如“名校背景”、“海归/海外工作经历”。若无相关数据则不提及。单人约 50 字，多人则每人约 25 字。

2. 能力诊断 (占比50%)：
   利用其管理的其他基金近一年周度数据。评估【TM模型-择时排名】与【TM模型-选股排名】，必须引用数据分位点（如“Top 10%”）来支撑观点。分析其业绩增长是源于精准择时，还是稳健的个股挖掘。

3. 策略独特性 (占比30%)：
   结合【SDI (选股驱动力)】指标分析其策略的独特程度。解释其收益与同类的相关性：高 SDI 意味着策略极具独特性/非共识，低相关性则反映了经理在获取 alpha 上的独立性。

评价语需专业且客观，严禁废话，直接给出数据背后的逻辑结论。"""),
        ("user", "以下是待分析的基金经理详细报告数据：\n\n{content}")
    ])

    # 3. 初始化模型
    llm = ChatOpenAI(
            model="deepseek-reasoner",
            openai_api_key="sk-84ec55da76944a109524df873d20d975",
            openai_api_base="https://api.deepseek.com",
        )

    # 4. 构建链
    chain = prompt | llm | StrOutputParser()

    # 5. 执行分析
    analysis_result = chain.invoke({"content": report_content})
    return analysis_result

In [3]:
# 使用示例
result = analyze_manager_report("000005_Analysis_Report.md")
print(result)

**经理画像：**
*   轩璇：13年固收领域经验，工学硕士背景，具备基金从业资格，属金融科班出身。
*   吴翠：14年证券从业经历，曾任信用评级机构副总，硕士研究生，同样为资深金融从业者。

**能力诊断：**
核心业绩源于突出的选股能力。在代表性混合基金中，轩璇（002222）与吴翠（001688）的TM选股排名长期稳居Top 20%甚至Top 10%分位（如近期16.9%、20.5%）。相比之下，择时能力（TMTiming）排名多在50%-80%区间，表现平平甚至偏弱。表明两者均是通过深度的个股挖掘而非精准的市场择时来驱动超额收益。

**策略独特性：**
吴翠在管混合基金（001688）的SDI高达0.79，显示其收益绝大部分由纯粹的选股驱动，策略极具独特性，与市场共识及同类基金的相关性低，获取Alpha的独立性很强。轩璇在混合策略上（SDI 0.50）也展现出一定的选股独特性，但在债券型产品中策略更偏向传统风格暴露。
